In [1]:
import keras

In [ ]:
batch_size = 32

In [ ]:
train_ds = keras.utils.text_dataset_from_directory( 
    '../../aclImdb/train/', 
    validation_split=0.2, 
    subset="training", 
    seed=123,
    batch_size=batch_size)

val_ds = keras.utils.text_dataset_from_directory( 
    '../../aclImdb/train/', 
    validation_split=0.2, 
    subset="validation", 
    seed=123, # same seed as above!
    batch_size=batch_size)

test_ds = keras.utils.text_dataset_from_directory( 
    '../../aclImdb/test/', 
    batch_size=batch_size)

In [ ]:

for inputs, targets in train_ds:
    print(inputs.shape, targets.shape)
    print(inputs.dtype, targets.dtype)
    print(inputs[0])
    print(targets[0])
    break

In [ ]:
from keras.layers import TextVectorization

max_tokens = 10000

vectorization_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode='multi_hot'
)


train_texts = train_ds.map(lambda x, y: x)
vectorization_layer.adapt(train_texts)


train_ds_bow = train_ds.map(lambda x, y: (vectorization_layer(x), y))
val_ds_bow = val_ds.map(lambda x, y: (vectorization_layer(x), y))
test_ds_bow = test_ds.map(lambda x, y: (vectorization_layer(x), y))

In [ ]:
for inputs, targets in train_ds_bow:
    print(inputs.shape, targets.shape)
    print(inputs.dtype, targets.dtype)
    print(inputs[0])
    print(targets[0])
    break

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout

In [ ]:
model = Sequential([
    Input(shape=(max_tokens,)),
    Dense(units=16, activation='relu'),
    Dropout(0.5),
    Dense(units=1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='rmsprop',
    metrics=['accuracy']
)

model.fit(
    train_ds_bow, 
    epochs=10, 
    validation_data=val_ds_bow
)

In [ ]:
print(f"Test accuracy: {model.evaluate(test_ds_bow)[1]:.4f}")

In [ ]:

vocabulary = vectorization_layer.get_vocabulary()
print(vocabulary[0:20]) # print 20 most common tokens

### Word Embeddings

In [ ]:
# Define the TextVectorization layer 
max_tokens = 10000 # Maximum vocabulary size 
sequence_length = 250 # Maximum sequence length 

vectorization_layer = TextVectorization( 
    max_tokens=max_tokens, 
    output_mode='int', 
    output_sequence_length=sequence_length )

train_texts = train_ds.map(lambda x, y: x) 
vectorization_layer.adapt(train_texts)

train_ds_int = train_ds.map(lambda x, y: (vectorization_layer(x), y)) 
val_ds_int = val_ds.map(lambda x, y: (vectorization_layer(x), y))
test_ds_int = test_ds.map(lambda x, y: (vectorization_layer(x), y))

In [ ]:
for inputs, targets in train_ds_int:
    print(inputs.shape, targets.shape)
    print(inputs.dtype, targets.dtype)
    print(inputs[0])
    print(targets[0])
    break

In [ ]:
from keras.layers import Embedding, Dropout, Reshape

embed_dim = 64 # dimension of the word embeddings

model = Sequential([
    Input(shape=(sequence_length,)),
    Embedding(input_dim=max_tokens, output_dim=embed_dim),
    Reshape((sequence_length * embed_dim,)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(units=1, activation='sigmoid')
])

model.summary()

model.compile(loss='binary_crossentropy', optimizer='rmsprop', metrics=['accuracy'])
model.fit(train_ds_int, epochs=10, validation_data=val_ds_int)

In [ ]:
# Extract the embedding weights from the trained model 
embedding_layer = model.layers[0] # the first layer of the model
embedding_weights = embedding_layer.get_weights()[0] # shape (10000, 64)

# Get the word index from the tokenizer 
vocabulary = vectorization_layer.get_vocabulary() 
index_word = {idx: word for idx, word in enumerate(vocabulary)}
word_index = {word: idx for idx, word in enumerate(vocabulary)}

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity 

# Function to find similar words 
def find_similar_words(target_word, top_n=10): 
    if target_word not in word_index: 
        return f'Word "{target_word}" not in vocabulary' 
    target_idx = word_index[target_word] 
    target_embedding = embedding_weights[target_idx].reshape(1, -1) # shape (1, 64)
    similarities = cosine_similarity(embedding_weights, target_embedding).reshape(-1) # shape (10000,)
    similar_indices = np.argsort(similarities)[-top_n-1:-1][::-1] 
    similar_words = [index_word[idx] for idx in similar_indices] 
    return similar_words 
    
word_to_check = 'bad'
print(f'Words similar to "{word_to_check}": {find_similar_words(word_to_check)}')